# Nemotron v1 SFT — Colab Pro+ A100

**Open in Colab Pro+, pick A100 GPU + High-RAM, then Run All.**

Repo: https://github.com/ykaya-jp/NVIDIA-Nemotron-Model-Reasoning-Challenge

Nemotron v1 SFT — Colab Pro+ A100 version.

> USAGE (Colab Pro+):
> 1. New Notebook → Runtime → Change runtime type → A100 (40 GB GPU,
>    High-RAM). Save & connect.
> 2. Paste this whole script into one cell.
> 3. Run All. Expected runtime: 6-10 h on A100 40 GB.
> 4. The trained LoRA adapter lands at
>    /content/drive/MyDrive/nemotron-2026/adapter_v1/. After it
>    finishes, download that folder (or upload it as a Kaggle Dataset
>    `ky7240/nemotron-v1-adapter`) and reference it from a thin
>    submission notebook (= GPU off, just writes submission.csv with
>    the adapter path) to submit to Kaggle.

ATTRIBUTION (= Apache 2.0 licensed public kernels we draw setup code
from; please credit when publishing):
- dgxchen/training-with-unsloth-to-achieve-0-85-lb (Kaggle kernel)
- konbu17/nemotron-sft-lora-with-cot (Kaggle kernel)
- huikang/end-to-end-finetuning-for-lb-0-85 (Kaggle kernel)

NOVEL CONTRIBUTION = the SFT data (data/processed/train_sft_verifier_
only.jsonl, 5418 records). It is generated by 5 deterministic Python
solvers in this repo (Roman / Physics / Unit / Cipher / Bit) producing
verifier-backed Chain-of-Thought traces. See
docs/strategy/winning-strategy.dense.md for the full rationale.

## Cell 1 — Drive mount + GitHub clone

In [ ]:
from google.colab import drive

drive.mount("/content/drive")

import os
import subprocess

REPO_URL = "https://github.com/ykaya-jp/NVIDIA-Nemotron-Model-Reasoning-Challenge.git"
REPO_DIR = "/content/nemotron"
if not os.path.isdir(REPO_DIR):
    subprocess.run(["git", "clone", "--depth", "1", REPO_URL, REPO_DIR], check=True)
os.chdir(REPO_DIR)
print("repo at", REPO_DIR)
subprocess.run(["git", "log", "-1", "--oneline"])

ADAPTER_DIR = "/content/drive/MyDrive/nemotron-2026/adapter_v1"
os.makedirs(ADAPTER_DIR, exist_ok=True)


## Cell 2 — Dependencies (plain transformers + peft + TRL stack)

In [ ]:
# Why no Unsloth: Unsloth issue #3480 (= "Cannot load Nvidia Nemotron
# Nano models") + a meta-tensor crash inside unsloth_zoo
# fix_untrained_tokens make the Unsloth fast-path unusable on this
# specific model. The NVIDIA / DataCamp reference notebook uses plain
# transformers + peft + TRL with the version pins below; this is the
# documented working stack for Nemotron-3-Nano-30B-A3B-BF16 on an
# A100 80 GB.
# References:
# - https://www.datacamp.com/tutorial/fine-tuning-nvidia-nemotron-3-nano
# - https://huggingface.co/nvidia/NVIDIA-Nemotron-3-Nano-30B-A3B-BF16
# - https://github.com/unslothai/unsloth/issues/3480
import subprocess
import sys


def _run(cmd):
    """Run shell command, stream output to the cell so the user sees errors."""
    print(f"$ {' '.join(cmd)}")
    subprocess.run(cmd, check=True)


# Step 1: build tooling for the Mamba CUDA extensions.
_run([sys.executable, "-m", "pip", "install", "-q", "-U", "packaging", "ninja"])

# Step 2: pin torch to the CUDA 12.8 build required by Nemotron-3-Nano.
# Colab default torch is usually fine, but the cu128 wheel is the one
# NVIDIA / Unsloth document for the Mamba kernels, so we force it.
_run([sys.executable, "-m", "pip", "uninstall", "-y",
      "torch", "torchvision", "torchaudio", "triton"])
_run([sys.executable, "-m", "pip", "install", "-q",
      "torch==2.7.1", "torchvision==0.22.1", "torchaudio==2.7.1",
      "--index-url", "https://download.pytorch.org/whl/cu128"])

# Step 3: HF stack with the exact pins from the NVIDIA / DataCamp
# reference. transformers 4.56.2 + trl 0.22.2 supports
# `completion_only_loss=True` and `processing_class=tokenizer`.
_run([sys.executable, "-m", "pip", "install", "-q", "-U",
      "transformers==4.56.2", "tokenizers", "trl==0.22.2",
      "accelerate", "datasets", "peft",
      "huggingface_hub", "safetensors",
      "sentencepiece", "nbformat"])

# Step 4: Mamba-SSM + causal-conv1d at the versions that ship in the
# NVIDIA reference. `--no-build-isolation` reuses the torch we just
# installed (otherwise pip spawns a clean env without torch and the
# CUDA build fails). First build ~5-10 min on A100; cached after.
_run([sys.executable, "-m", "pip", "install", "-q",
      "-U", "--no-build-isolation",
      "mamba_ssm==2.2.5", "causal_conv1d==1.5.2"])

import torch

print(
    "cuda:",
    torch.cuda.is_available(),
    "device:",
    torch.cuda.get_device_name(0) if torch.cuda.is_available() else "cpu",
    "mem GB:",
    torch.cuda.get_device_properties(0).total_memory / 1024**3 if torch.cuda.is_available() else 0,
)


## Cell 3 — Load base model + tokenizer (plain transformers path)

In [ ]:
# We DO NOT wrap with peft here. SFTTrainer accepts `peft_config=...`
# and applies LoRA internally — this avoids the meta-tensor edge case
# we hit when wrapping the model first and then letting TRL re-prepare
# it. attn_implementation="eager" is required: the hybrid Mamba layers
# refuse SDPA / FA-2 paths.
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer

MAX_SEQ_LEN = 2048   # = 95th-percentile of our SFT records; 4096 risks
                     #   OOM at 30B + bs1 + LoRA on A100-80 GB. Inference
                     #   side still uses 4096 (we just train on shorter
                     #   contexts).
MODEL_NAME = "nvidia/NVIDIA-Nemotron-3-Nano-30B-A3B-BF16"

tokenizer = AutoTokenizer.from_pretrained(
    MODEL_NAME,
    trust_remote_code=True,
    use_fast=True,
)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "right"

model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    trust_remote_code=True,
    dtype=torch.bfloat16,
    device_map="auto",
    attn_implementation="eager",
)
model.gradient_checkpointing_enable()
model.config.use_cache = False
model.config.pad_token_id = tokenizer.pad_token_id
model.config.eos_token_id = tokenizer.eos_token_id
print("✓ base model loaded (BF16, eager attn, gradient checkpointing on)")


## Cell 4 — Load SFT data + chat template + upsample + completion-only

In [ ]:
import json
from collections import Counter

from datasets import Dataset, concatenate_datasets

SFT_PATH = os.path.join(REPO_DIR, "data/processed/train_sft_verifier_only.jsonl")
assert os.path.exists(SFT_PATH), f"SFT data missing at {SFT_PATH}"
rows = [json.loads(l) for l in open(SFT_PATH)]
print(f"loaded {len(rows)} verifier-backed records")
print("source distribution:", Counter(r["source"] for r in rows))

INFERENCE_USER_SUFFIX = (
    "\nPlease put your final answer inside `\\boxed{}`. For example: `\\boxed{your answer}`"
)


def render(rec):
    user_content = rec["user"] + INFERENCE_USER_SUFFIX
    prompt_text = tokenizer.apply_chat_template(
        [{"role": "user", "content": user_content}],
        tokenize=False,
        add_generation_prompt=True,
        enable_thinking=True,
    )
    return {"text": prompt_text + rec["assistant"]}


ds = Dataset.from_list([render(r) for r in rows])

# Codex review 3 §3 — upsample under-represented bit (× 13) and cipher
# (× 2) so they survive epoch-1 gradient mass.
upsample = {"solver_bit": 13, "solver_cipher": 2}
extras = []
for r in rows:
    n = upsample.get(r["source"], 1)
    if n > 1:
        extras.extend([r] * (n - 1))
if extras:
    extras_ds = Dataset.from_list([render(r) for r in extras])
    ds = concatenate_datasets([ds, extras_ds]).shuffle(seed=42)
print(f"after upsampling: {len(ds)} records")


## Cell 5 — SFT training (1 epoch, completion-only loss, LoRA via TRL)

In [ ]:
# We let SFTTrainer apply the LoRA adapter internally (peft_config=...)
# rather than wrapping with get_peft_model() ourselves. This is the
# code path the NVIDIA / DataCamp reference uses and it sidesteps the
# meta-tensor crash that Unsloth's fix_untrained_tokens triggers.
# target_modules="all-linear" matches NVIDIA's recommendation;
# completion_only_loss=True masks the prompt and only learns on the
# assistant turn (= the verifier-backed CoT). adamw_torch_fused is
# faster than adamw_8bit on A100 and avoids the bitsandbytes
# dependency.
from peft import LoraConfig
from trl import SFTConfig, SFTTrainer

lora_config = LoraConfig(
    r=32,
    lora_alpha=64,
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM",
    target_modules="all-linear",
)

args = SFTConfig(
    output_dir="/content/checkpoints",
    per_device_train_batch_size=1,
    gradient_accumulation_steps=32,         # effective batch = 32
    num_train_epochs=1,
    learning_rate=2e-4,
    warmup_ratio=0.03,
    weight_decay=0.0,
    lr_scheduler_type="linear",
    bf16=True,
    fp16=False,
    tf32=True,
    gradient_checkpointing=True,
    logging_steps=20,
    save_strategy="steps",
    save_steps=2000,
    save_total_limit=1,
    optim="adamw_torch_fused",
    seed=42,
    report_to="none",
    dataset_text_field="text",
    max_length=MAX_SEQ_LEN,                 # TRL >=0.22 renamed max_seq_length -> max_length
    packing=False,
    completion_only_loss=True,
    remove_unused_columns=False,
    dataloader_num_workers=2,
)

trainer = SFTTrainer(
    model=model,
    args=args,
    train_dataset=ds,
    peft_config=lora_config,
    processing_class=tokenizer,             # replaces the deprecated `tokenizer=` arg
)
trainer.train()
print("✓ training done")


## Cell 6 — Save adapter to Drive

In [ ]:
# After training the model object is a PeftModel wrapping the base.
# save_pretrained writes ONLY the LoRA adapter (~200-400 MB), which is
# what the Kaggle metric notebook expects.
trainer.model.save_pretrained(ADAPTER_DIR)
tokenizer.save_pretrained(ADAPTER_DIR)
print("saved adapter to", ADAPTER_DIR)
for fn in sorted(os.listdir(ADAPTER_DIR)):
    sz = os.path.getsize(os.path.join(ADAPTER_DIR, fn))
    print(f"  {fn}: {sz:,} bytes")
print()
print("Next: upload this folder to Kaggle as `ky7240/nemotron-v1-adapter`,")
print("then push the thin submission notebook (= GPU off) to Kaggle and submit.")
